# 🔍 SHAP 기반 XAI (설명 가능한 AI)

> **학습 목표**: SHAP을 사용하여 머신러닝 모델의 예측을 해석하고 설명합니다.

---

## 📋 학습 내용

1. ✅ SHAP (SHapley Additive exPlanations) 개념
2. ✅ TreeExplainer로 랜덤포레스트 해석
3. ✅ Force Plot - 개별 예측 설명
4. ✅ Summary Plot - 전역적 중요도
5. ✅ Dependence Plot - 피처 관계 분석
6. ✅ Feature Importance 비교 (SHAP vs 기본)

**소요 시간**: 약 30분  
**난이도**: ⭐⭐ (중급)  
**사전 지식**: scikit-learn 기초, 랜덤포레스트

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 데이터 분석
import pandas as pd
import numpy as np

# 머신러닝
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# SHAP - XAI
try:
    import shap
    print(f"✅ SHAP 버전: {shap.__version__}")
except ImportError:
    print("❌ SHAP 설치 필요: pip install shap")

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

# SHAP 초기화 (JavaScript 시각화)
try:
    shap.initjs()
except:
    pass

print("✅ 라이브러리 로드 완료!")

## 📂 Step 2: KAMP 데이터 로드

In [ ]:
# ============================================================
# KAMP 실데이터 경로 (dataset/ 디렉토리)
# ============================================================
kamp_data_dir = Path('../../dataset/part1-1')

# 1순위: 에스케이지 (26컬럼 → SHAP Feature Importance에 최적)
kamp_esg = kamp_data_dir / '제조AI데이터셋_에스케이지 주식회사.csv'
# 2순위: 진부 (앞 실습과 연속성)
kamp_jinbu = kamp_data_dir / '진부_공개용데이터_V1.csv'
# Fallback: 샘플 데이터
sample_path = Path('../data/sample_kamp_cnc.csv')

# 데이터 로드 함수
def load_data(file_path):
    encodings = ['utf-8-sig', 'cp949', 'utf-8']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ 데이터 로드 성공! (인코딩: {encoding})")
            return df
        except:
            continue
    raise ValueError("데이터 로드 실패")

# ============================================================
# 데이터 로드: 에스케이지 → 진부 → 샘플 fallback
# ============================================================
if kamp_esg.exists():
    df = load_data(kamp_esg)
    data_source = 'KAMP 에스케이지 (26컬럼 - SHAP 최적)'
    # 대용량 데이터 샘플링 (메모리 절약)
    if len(df) > 50000:
        df = df.sample(n=50000, random_state=42).reset_index(drop=True)
        print(f"  📊 대용량 데이터 샘플링: 50,000행 사용")
elif kamp_jinbu.exists():
    df = load_data(kamp_jinbu)
    data_source = 'KAMP 진부 (15컬럼)'
elif sample_path.exists():
    df = load_data(sample_path)
    data_source = '샘플 데이터 (fallback)'
    print("⚠️ KAMP 실데이터를 찾을 수 없어 샘플 데이터를 사용합니다.")
else:
    raise FileNotFoundError("❌ 데이터 파일을 찾을 수 없습니다.")

print(f"\n📊 데이터 소스: {data_source}")
print(f"📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"💾 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 데이터 미리보기
df.head()

In [ ]:
# 수치형 컬럼 선택 (모델링용)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"📊 수치형 컬럼: {len(numeric_cols)}개")
print(numeric_cols)

# 결측치 확인
missing = df[numeric_cols].isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠️ 결측치 발견:")
    print(missing[missing > 0])
    # 결측치 제거
    df = df.dropna(subset=numeric_cols)
    print(f"✅ 결측치 제거 후: {df.shape[0]:,}행")
else:
    print("\n✅ 결측치 없음")

## 🤖 Step 3: 머신러닝 모델 학습

**목표**: SHAP 해석을 위한 RandomForest 회귀 모델 학습

In [ ]:
# 피처(X)와 타겟(y) 분리
# 첫 번째 컬럼을 타겟으로 사용 (또는 사용자가 지정)
target_col = numeric_cols[0]
feature_cols = numeric_cols[1:]

print(f"🎯 타겟 변수: {target_col}")
print(f"📊 피처 변수: {len(feature_cols)}개")
print(f"   {feature_cols[:5]}...")

X = df[feature_cols]
y = df[target_col]

# 데이터 분할 (Train 80%, Test 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n📦 학습 데이터: {X_train.shape[0]:,}개")
print(f"📦 테스트 데이터: {X_test.shape[0]:,}개")

In [ ]:
# RandomForest 모델 학습
print("🔄 RandomForest 모델 학습 중...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 모델 성능 평가
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"\n✅ 모델 학습 완료!")
print(f"\n📊 성능 지표:")
print(f"   Train R²: {train_r2:.4f} | RMSE: {train_rmse:.4f}")
print(f"   Test R²:  {test_r2:.4f} | RMSE: {test_rmse:.4f}")

if train_r2 - test_r2 > 0.1:
    print(f"\n⚠️ 과적합 가능성 (Train-Test R² 차이: {train_r2 - test_r2:.4f})")
else:
    print(f"\n✅ 적절한 일반화 성능")

## 🔍 Step 4: SHAP TreeExplainer

**TreeExplainer**: 트리 기반 모델(RandomForest, XGBoost, LightGBM)에 최적화된 SHAP Explainer

In [ ]:
# SHAP TreeExplainer 생성
print("🔄 SHAP TreeExplainer 생성 중...")

explainer = shap.TreeExplainer(model)

# SHAP 값 계산 (테스트 데이터에 대해)
print("🔄 SHAP 값 계산 중... (시간이 걸릴 수 있습니다)")

# 전체 데이터가 크면 샘플링
sample_size = min(1000, len(X_test))
X_test_sample = X_test.iloc[:sample_size]

shap_values = explainer.shap_values(X_test_sample)

print(f"\n✅ SHAP 값 계산 완료!")
print(f"   SHAP 값 형태: {shap_values.shape}")

# expected_value가 배열인 경우 처리 (RandomForest 등)
base_val = explainer.expected_value
if hasattr(base_val, '__len__'):
    base_val = float(np.mean(base_val))
print(f"   기준값 (Expected Value): {base_val:.4f}")

## 📊 Step 5: SHAP 시각화

### 5.1 Summary Plot (전역적 중요도)

In [ ]:
# Summary Plot - 전체 피처의 중요도와 영향
print("📊 SHAP Summary Plot 생성...")

plt.figure(figsize=(14, 8))
shap.summary_plot(shap_values, X_test_sample, plot_type="dot", show=False)
plt.title('SHAP Summary Plot - 전역적 피처 중요도', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 해석 가이드:")
print("   - 위쪽: 중요도가 높은 피처")
print("   - 빨간색: 높은 피처 값")
print("   - 파란색: 낮은 피처 값")
print("   - 오른쪽: 예측값 증가에 기여")
print("   - 왼쪽: 예측값 감소에 기여")

### 5.2 Bar Plot (평균 절대 SHAP 값)

In [ ]:
# Bar Plot - 평균 절대 SHAP 값
plt.figure(figsize=(14, 6))
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", show=False)
plt.title('SHAP Bar Plot - 평균 절대 중요도', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 상위 10개 피처 추출
feature_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': X_test_sample.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\n📊 상위 10개 중요 피처:")
print(importance_df.head(10).to_string(index=False))

### 5.3 Force Plot (개별 예측 설명)

In [ ]:
# Force Plot - 첫 번째 테스트 샘플 설명
print("📊 SHAP Force Plot 생성 (첫 번째 샘플)...")

# 개별 예측 설명
sample_idx = 0
sample_shap = shap_values[sample_idx]
sample_data = X_test_sample.iloc[sample_idx]
actual_value = y_test.iloc[sample_idx]
predicted_value = model.predict(sample_data.values.reshape(1, -1))[0]

# expected_value가 배열인 경우 처리 (RandomForest 등)
base_val = explainer.expected_value
if hasattr(base_val, '__len__'):
    base_val = float(np.mean(base_val))

print(f"\n🎯 샘플 #{sample_idx} 분석:")
print(f"   실제 값: {actual_value:.4f}")
print(f"   예측 값: {predicted_value:.4f}")
print(f"   기준 값: {base_val:.4f}")
print(f"   오차: {abs(actual_value - predicted_value):.4f}")

# Force Plot 시각화
try:
    shap.force_plot(
        base_val,
        sample_shap,
        sample_data,
        matplotlib=True,
        show=False
    )
    plt.title(f'SHAP Force Plot - 샘플 #{sample_idx} 예측 설명', fontsize=12, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()
except:
    print("⚠️ Force Plot 시각화 실패 (matplotlib 모드)")

print("\n💡 해석 가이드:")
print("   - 빨간색: 예측값을 증가시키는 피처")
print("   - 파란색: 예측값을 감소시키는 피처")
print("   - 화살표 길이: 기여도 크기")

### 5.4 Dependence Plot (피처 관계 분석)

In [ ]:
# Dependence Plot - 가장 중요한 피처 선택
top_feature = importance_df.iloc[0]['feature']

print(f"📊 SHAP Dependence Plot 생성 (피처: {top_feature})...")

plt.figure(figsize=(14, 6))
shap.dependence_plot(
    top_feature,
    shap_values,
    X_test_sample,
    show=False
)
plt.title(f'SHAP Dependence Plot - {top_feature}', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 해석 가이드:")
print("   - X축: 피처 값")
print("   - Y축: SHAP 값 (예측에 대한 기여도)")
print("   - 색상: 상호작용하는 다른 피처의 값")
print(f"   - 경향: {top_feature} 값이 증가할 때 예측값에 미치는 영향")

## 🔄 Step 6: Feature Importance 비교

**SHAP vs RandomForest 기본 Feature Importance**

In [ ]:
# RandomForest 기본 Feature Importance
rf_importance = pd.DataFrame({
    'feature': X_test_sample.columns,
    'rf_importance': model.feature_importances_
}).sort_values('rf_importance', ascending=False)

# SHAP Feature Importance
shap_importance = importance_df.copy()
shap_importance.columns = ['feature', 'shap_importance']

# 병합
comparison = rf_importance.merge(shap_importance, on='feature')

# 정규화 (0-1 범위)
comparison['rf_norm'] = comparison['rf_importance'] / comparison['rf_importance'].max()
comparison['shap_norm'] = comparison['shap_importance'] / comparison['shap_importance'].max()

print("📊 Feature Importance 비교 (상위 10개):")
print(comparison.head(10).to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RandomForest Importance
top_n = 15
rf_top = comparison.head(top_n).sort_values('rf_importance')
axes[0].barh(rf_top['feature'], rf_top['rf_importance'], color='skyblue')
axes[0].set_title('RandomForest Feature Importance', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')
axes[0].grid(alpha=0.3)

# SHAP Importance
shap_top = comparison.head(top_n).sort_values('shap_importance')
axes[1].barh(shap_top['feature'], shap_top['shap_importance'], color='coral')
axes[1].set_title('SHAP Feature Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mean |SHAP Value|')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 차이점:")
print("   - RF Importance: 모델 내부 불순도 감소 기반 (글로벌)")
print("   - SHAP Importance: 실제 예측 기여도 기반 (로컬 + 글로벌)")
print("   - SHAP은 개별 예측에 대한 설명 가능")

## 💾 Step 7: 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# Feature Importance 저장
importance_file = output_dir / '03_shap_feature_importance.csv'
comparison.to_csv(importance_file, index=False, encoding='utf-8-sig')
print(f"✅ Feature Importance 저장: {importance_file}")

# SHAP 값 저장 (numpy)
shap_file = output_dir / '03_shap_values.npy'
np.save(shap_file, shap_values)
print(f"✅ SHAP 값 저장: {shap_file}")

# 모델 성능 저장
performance = pd.DataFrame({
    'metric': ['Train R²', 'Test R²', 'Train RMSE', 'Test RMSE'],
    'value': [train_r2, test_r2, train_rmse, test_rmse]
})
performance_file = output_dir / '03_model_performance.csv'
performance.to_csv(performance_file, index=False, encoding='utf-8-sig')
print(f"✅ 모델 성능 저장: {performance_file}")

print("\n🎉 SHAP 분석 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. SHAP TreeExplainer로 RandomForest 모델 해석
2. Summary Plot으로 전역적 피처 중요도 분석
3. Force Plot으로 개별 예측 설명
4. Dependence Plot으로 피처 관계 분석
5. SHAP vs RF Feature Importance 비교

### 💡 핵심 인사이트
- **SHAP의 장점**:
  - 개별 예측에 대한 설명 가능 (Local Interpretability)
  - 이론적 근거가 명확 (Shapley Value)
  - 모델 종류에 관계없이 일관된 해석 제공
  
- **활용 방법**:
  - Summary Plot: 전체적인 피처 중요도 파악
  - Force Plot: 특정 예측 결과 설명 (고객 설명용)
  - Dependence Plot: 비선형 관계 발견

- **실무 적용**:
  - 모델 디버깅: 이상한 예측 원인 파악
  - 피처 엔지니어링: 중요 피처 기반 새로운 변수 생성
  - 의사결정 지원: 예측 근거 제시로 신뢰도 향상

### 📚 다음 단계
- **Part 1-2**: 이미지 데이터 전처리 및 불량 분류
- **Part 2-1**: FFT 주파수 분석 및 이상 탐지

### 🔗 참고 자료
- [SHAP 공식 문서](https://shap.readthedocs.io/)
- [SHAP GitHub](https://github.com/slundberg/shap)
- [Shapley Value 논문](https://arxiv.org/abs/1705.07874)

---

*제조AI 교육 v12 Enhanced | Part 1-1 | 2025.02*